[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/session2_logistic_map_bridge.ipynb)


# What One Small Change Can Do — Simulating the Logistic Map

This notebook is not a math lesson. It's a series of small experiments, all built from the exact
same one-line rule. Each "Take" changes **one thing**, simulates, and shows you what happened.
No new theory needed to follow along -- just watch what varying one detail actually does.

**Why this matters for the course:** Kollman & Page's seven concepts (adaptation, difference,
externalities, path dependence, geography, networks, emergence) all assume many agents
interacting. Everything in this notebook has **zero agents** -- just one number, changing.
That's the point: some of the strangest behavior in complexity science doesn't need a social
system at all. Watch for the parts that rhyme with things we'll see K&P describe socially --
and the parts that don't.


First, the tools we'll need throughout: `numpy` for the math, `pandas` for a table or two, and
`matplotlib` for every plot.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True

print("Ready.")


---
## Take 1 — The rule, doing nothing interesting yet

Before we simulate anything, let's just look at what we're working with: one rule, applied once.
Give it a number between 0 and 1, it gives you another number between 0 and 1 back.


Here's the whole rule, as a function. `x` is the number we feed in, `r` is a dial we'll be turning
throughout this notebook -- for now, just think of it as a fixed setting.


In [ ]:
def logistic_step(x, r):
    """Feed in x, get the next value back. r is just a dial we can turn."""
    return r * x * (1 - x)


Let's use it: plug in ten evenly-spaced `x` values, all at one fixed setting of `r`, and put the
input and output side by side in a table.


In [ ]:
r = 0.8
x_values = np.linspace(0, 1, 10)

table = pd.DataFrame({"x_t": x_values, "x_next": logistic_step(x_values, r)})
table


Ten rows, ten pairs of numbers -- notice the output isn't a simple multiple of the input; it rises
then falls back toward zero as `x_t` approaches 1. Let's see that same relationship as a smooth
curve instead of a table, using many more points.


In [ ]:
x_grid = np.linspace(0, 1, 300)

plt.figure()
plt.plot(x_grid, logistic_step(x_grid, r))
plt.title(f"The rule, r = {r}")
plt.xlabel("x_t"); plt.ylabel("x_next")
plt.show()


A table, and a smooth curve -- nothing surprising here. One input, one output, every time, no
ambiguity. This is our baseline.

This rule -- $x_{t+1} = r \cdot x_t(1-x_t)$ -- is called the **logistic map**. Every Take from here
on reuses this exact same rule, unchanged. Only *how we use it* will change, one detail at a time.


---
## Take 2 — What if we feed the output back in, over and over?

Here's the one change: instead of using the rule once, we're going to use its own output as the
next input, again and again.


This function does exactly that: start at some `x0`, and repeatedly replace it with
`logistic_step` applied to itself, keeping every value along the way.


In [ ]:
def simulate(r, x0, n_steps):
    """Start at x0, and keep feeding each result back in as the next input."""
    xs = [x0]
    for _ in range(n_steps - 1):
        xs.append(logistic_step(xs[-1], r))
    return np.array(xs)


Let's try it from four very different starting points, using the exact same rule (`r = 2.5`) and
the exact same number of repetitions, and watch where each one ends up.


In [ ]:
r = 2.5

plt.figure()
for x0 in [0.05, 0.3, 0.7, 0.95]:
    plt.plot(simulate(r, x0, n_steps=25), marker="o", markersize=3, label=f"start = {x0}")
plt.title(f"Same rule (r = {r}), four different starting points")
plt.xlabel("step"); plt.ylabel("x")
plt.legend()
plt.show()


**AWE:** four wildly different starting points -- and they all end up in the exact same place.
The system doesn't care where it started. Whatever happened before doesn't matter -- only the rule
and where you are *right now* determine where you go next.

That settled-on value is called an **attractor** -- specifically, since it's a single fixed value,
a *fixed-point attractor*. And the fact that every starting point gets pulled into the same one is
what it means for a system to **forget its initial condition**.


---
## Take 3 — What if we turn the dial up a little?

`r` is the one number in our rule we haven't touched yet -- think of it as a dial. Right now it's
at 2.5. Let's turn it up to 3.2, change nothing else, and run the exact same repeated-feeding
experiment as Take 2.


Same `simulate` function as before, same style of plot -- only the value of `r` is different.


In [ ]:
r = 3.2

plt.figure()
plt.plot(simulate(r, x0=0.3, n_steps=30), marker="o", markersize=3)
plt.title(f"r = {r}")
plt.xlabel("step"); plt.ylabel("x")
plt.show()


**AWE:** it stopped settling on one value. Now it bounces forever between two values -- forever
visiting the same two numbers, back and forth, never landing on just one. We turned one dial by
0.7, and the system's entire long-run behavior changed in *kind*, not just in degree: one attractor
became two.

The point where that split happens, as we turn the dial, is called a **bifurcation** -- and the new
two-value pattern it settles into is called a *period-2 attractor*.


---
## Take 4 — What if we keep turning that same dial, systematically?

Take 3 was one nudge, one result. This time, let's automate it: run that exact same
settle-and-record experiment at thousands of different `r` values in a row, sweeping smoothly from
2.5 up to 4.0.


This function runs many `r` values *simultaneously*, as arrays, rather than one at a time. It
first lets every trajectory settle (and throws those early steps away), then records what each one
actually settles onto.


In [ ]:
def sweep(r_min, r_max, n_r, n_settle=300, n_record=150, x0=0.2):
    """Run the same simulate-and-throw-away-the-start idea, for many r values
    at once, and record only where each one ends up long-run."""
    r_values = np.linspace(r_min, r_max, n_r)
    x = np.full(n_r, x0)

    for _ in range(n_settle):          # let each r's trajectory settle first
        x = logistic_step(x, r_values)

    all_r, all_x = [], []
    for _ in range(n_record):          # now record where it actually lands
        x = logistic_step(x, r_values)
        all_r.append(r_values)
        all_x.append(x)

    return np.concatenate(all_r), np.concatenate(all_x)


Let's run it across the full range and plot every `r` against everywhere it ends up settling.


In [ ]:
bif_r, bif_x = sweep(r_min=2.5, r_max=4.0, n_r=2000)

plt.figure()
plt.scatter(bif_r, bif_x, s=0.15, alpha=0.4, color="black")
plt.title("Every r, where it settles")
plt.xlabel("r"); plt.ylabel("long-run x")
plt.show()


**AWE:** one settling point becomes two, becomes four, becomes eight -- faster and faster --
until it dissolves into what looks like static, with a few strange clear gaps still visible inside
the static. Nobody told the system to do this. It's a direct, mechanical consequence of turning one
dial continuously on one unchanged, one-line rule. No randomness anywhere in this code -- every
pixel of this picture is fully determined by the rule alone.

This whole picture is called a **bifurcation diagram**, and the region where it turns to static is
called **chaos**.


---
## Take 5 — What if we barely change where we start, instead of the rule?

Different kind of change this time: leave `r` fixed, and instead nudge the *starting point* by an
almost unimaginably small amount -- one hundred-millionth.


This function runs `simulate` twice from two almost-identical starting points, so we can compare
them directly.


In [ ]:
def two_starts(r, x0, epsilon, n_steps):
    """Run the simulation twice, from two almost-identical starting points."""
    x1 = simulate(r, x0, n_steps)
    x2 = simulate(r, x0 + epsilon, n_steps)
    return x1, x2


Let's compare it at two different `r` values side by side: one we already know settles quietly
(`r = 2.9`), and one deep in the chaotic region from Take 4 (`r = 3.9`).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for r_val, ax in zip([2.9, 3.9], axes):
    x1, x2 = two_starts(r=r_val, x0=0.2, epsilon=1e-8, n_steps=60)
    ax.plot(x1, label="start A")
    ax.plot(x2, linestyle="--", label="start B (barely different)")
    ax.set_title(f"r = {r_val}")
    ax.legend()

plt.tight_layout()
plt.show()


**AWE:** at `r = 2.9`, the two lines are indistinguishable -- the tiny difference we started with
just vanished, the same forgetting we saw back in Take 2. But at `r = 3.9`, they track together for
a while, and then split completely apart. Same rule, virtually the same starting point, totally
different outcome.

This is called **sensitivity to initial conditions**, and it's the defining signature of chaos:
instead of forgetting small differences, the system **remembers and amplifies** them.


---
## Take 6 — What if we put a number on how fast that split happens?

Take 5 showed us divergence by eye, at two specific `r` values. Let's turn that into a single
measurable number, computed directly from the rule itself.


This function tracks, at every step, how strongly the rule stretches or shrinks a tiny difference
(its slope), and averages that over many steps -- after first letting the trajectory settle.


In [ ]:
def divergence_speed(r, x0=0.4, burn_in=200, n_steps=2000):
    """On average, does a tiny error grow or shrink at each step? One number
    answers this -- computed straight from the rule's own slope."""
    x = x0
    for _ in range(burn_in):
        x = logistic_step(x, r)

    total = 0.0
    for _ in range(n_steps):
        slope_here = abs(r * (1 - 2*x))
        total += np.log(slope_here + 1e-16)
        x = logistic_step(x, r)

    return total / n_steps


Now let's sweep this across the exact same range of `r` we used for Take 4's bifurcation diagram,
and plot the two, one above the other, so we can line them up directly.


In [ ]:
r_sweep = np.linspace(2.5, 4.0, 400)
speed = np.array([divergence_speed(r) for r in r_sweep])

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].scatter(bif_r, bif_x, s=0.1, alpha=0.3, color="black")
axes[0].set_ylabel("long-run x")
axes[0].set_title("Take 4's picture (top) vs. Take 6's number (bottom)")

axes[1].plot(r_sweep, speed, color="tab:red")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("divergence speed")
axes[1].set_xlabel("r")
plt.tight_layout()
plt.show()


**AWE:** this one number goes negative exactly where Take 4's picture shows one settled value, and
crosses zero exactly where the picture splits. One quantity, computed from nothing but the rule
itself, tells you *in advance* whether tiny errors will shrink or explode -- without ever having to
run the two-trajectory experiment from Take 5 at all.

This number is called the **Lyapunov exponent** ($\lambda$). Negative means predictable, positive
means chaotic, and zero marks the exact boundary -- a bifurcation point, same as Take 3.

**Two concrete numbers worth sitting with, from this same computation:**
- At `r = 2.9` ($\lambda < 0$): a small error shrinks by about 10% every step -- cut in half within
  6 steps.
- At `r = 3.7` ($\lambda > 0$): a small error *grows* by about 43% every step -- doubled in under 2
  steps, and ten times larger within 6 steps.

Past that threshold, no amount of better data or a fancier model buys you longer-range prediction
-- it's mathematically, not practically, out of reach. The honest move shifts from *predict further
ahead* to *react faster*.


---
## What we actually did, across six takes

| Take | The one thing we changed | What it revealed |
|---|---|---|
| 1 | nothing -- just looked at the rule | a plain curve |
| 2 | fed the output back in, repeatedly | the system **forgets** where it started |
| 3 | nudged `r` up a little | one settled value became two |
| 4 | swept `r` across a whole range | the full cascade: 1 -> 2 -> 4 -> 8 -> chaos |
| 5 | nudged the *starting point*, not `r` | the system **remembers and amplifies** tiny differences |
| 6 | measured Take 5's effect directly | one number predicts predictability itself |

Every single Take reused the same one-line rule from Take 1. Nothing about the rule ever changed
-- only how we *used* it. That's the whole lesson: complexity, unpredictability, sudden regime
change -- none of it required more variables, more agents, or more randomness. It required
**simulating and watching**, one change at a time.

**Where K&P picks this up:** their seven concepts (adaptation, difference, externalities, path
dependence, geography, networks, emergence) bring in the thing this notebook never had at all --
many different agents, interacting with each other. Some of what you just watched will rhyme with
what's coming (settling vs. not settling looks a lot like path dependence; Take 4's cascade shares
a name, "emergence," with K&P's very different, agent-based version). Some of it won't rhyme at
all -- and it's worth staying alert to which is which.
